# MIRACL
## Metadata Indexing and Retrieval for Automated Context Labelling

MIRACL is a document retrieval and question-answering system designed
to process complex PDF documents, preserve document structure and metadata,
perform semantic retrieval, and generate grounded answers using RAG.

### Pipeline

PDF
↓
Adobe PDF Extract API
↓
Structured JSON
↓
Metadata Extraction
↓
Contextual Chunking
↓
BGE Embeddings
↓
ChromaDB
↓
LlamaIndex Retrieval
↓
LangChain
↓
Ollama / Llama 3.2
↓
Grounded Answer

In [2]:
from importlib.metadata import version

print("Adobe PDF Services SDK:", version("pdfservices-sdk"))
print("sentence-transformers:", version("sentence-transformers"))
print("chromadb:", version("chromadb"))
print("llama-index:", version("llama-index"))
print("llama-index-vector-stores-chroma:", version("llama-index-vector-stores-chroma"))
print("llama-index-embeddings-huggingface:", version("llama-index-embeddings-huggingface"))
print("langchain:", version("langchain"))
print("langchain-ollama:", version("langchain-ollama"))

print("\nAll required MIRACL packages verified successfully.")

Adobe PDF Services SDK: 4.3.0
sentence-transformers: 6.0.1
chromadb: 1.5.9
llama-index: 0.14.24
llama-index-vector-stores-chroma: 0.6.0
llama-index-embeddings-huggingface: 0.8.0
langchain: 1.4.1
langchain-ollama: 1.1.0

All required MIRACL packages verified successfully.


In [3]:
# ============================================
# MIRACL - Imports
# ============================================

import os
import io
import json
import zipfile
import requests
from pathlib import Path

# Adobe PDF Extract
from adobe.pdfservices.operation.auth.service_principal_credentials import (
    ServicePrincipalCredentials,
)
from adobe.pdfservices.operation.pdf_services import PDFServices
from adobe.pdfservices.operation.pdf_services_media_type import (
    PDFServicesMediaType,
)
from adobe.pdfservices.operation.pdfjobs.jobs.extract_pdf_job import (
    ExtractPDFJob,
)
from adobe.pdfservices.operation.pdfjobs.params.extract_pdf.extract_element_type import (
    ExtractElementType,
)
from adobe.pdfservices.operation.pdfjobs.params.extract_pdf.extract_pdf_params import (
    ExtractPDFParams,
)
from adobe.pdfservices.operation.pdfjobs.result.extract_pdf_result import (
    ExtractPDFResult,
)

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# LlamaIndex
from llama_index.core import Document, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

# LangChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

print("All MIRACL imports successful.")

All MIRACL imports successful.


In [4]:
# ============================================
# MIRACL - Adobe PDF Services Credentials
# ============================================

from getpass import getpass

ADOBE_CLIENT_ID = input("Enter Adobe Client ID: ")
ADOBE_CLIENT_SECRET = getpass("Enter Adobe Client Secret: ")

print("\nAdobe credentials received.")
print("Client ID:", "Configured" if ADOBE_CLIENT_ID else "Missing")
print("Client Secret:", "Configured" if ADOBE_CLIENT_SECRET else "Missing")

Enter Adobe Client ID:  950294ffef71422b82c4b1b2ee2e1136
Enter Adobe Client Secret:  ········



Adobe credentials received.
Client ID: Configured
Client Secret: Configured


In [7]:
# ============================================
# MIRACL - Test Adobe PDF Services Client
# ============================================

credentials = ServicePrincipalCredentials(
    client_id=ADOBE_CLIENT_ID,
    client_secret=ADOBE_CLIENT_SECRET,
)

pdf_services = PDFServices(
    credentials=credentials
)

print("Adobe PDF Services client initialized successfully.")

Adobe PDF Services client initialized successfully.


In [8]:
# ============================================
# MIRACL - Select PDF by Filename
# ============================================

from pathlib import Path

pdf_filename = input("Enter the PDF filename: ").strip()

pdf_path = Path(pdf_filename)

if not pdf_path.exists():
    raise FileNotFoundError(
        f"PDF not found: {pdf_filename}\n"
        "Make sure the PDF is in the same folder as this notebook."
    )

if pdf_path.suffix.lower() != ".pdf":
    raise ValueError("Please provide a PDF filename ending with .pdf")

print("PDF selected successfully.")
print("File name:", pdf_path.name)
print("File size:", round(pdf_path.stat().st_size / 1024, 2), "KB")

Enter the PDF filename:  reume.pdf


PDF selected successfully.
File name: reume.pdf
File size: 98.74 KB


In [10]:
# ============================================
# MIRACL - Upload PDF to Adobe
# ============================================

with open(pdf_path, "rb") as file:
    input_stream = file.read()

input_asset = pdf_services.upload(
    input_stream=input_stream,
    mime_type=PDFServicesMediaType.PDF,
)

print("PDF uploaded to Adobe successfully.")
print("Adobe asset created:", input_asset)

PDF uploaded to Adobe successfully.
Adobe asset created: <adobe.pdfservices.operation.io.cloud_asset.CloudAsset object at 0x16181b380>


In [11]:
# ============================================
# MIRACL - Create & Submit Extraction Job
# ============================================

extract_pdf_params = ExtractPDFParams(
    elements_to_extract=[
        ExtractElementType.TEXT,
    ]
)

extract_pdf_job = ExtractPDFJob(
    input_asset=input_asset,
    extract_pdf_params=extract_pdf_params,
)

location = pdf_services.submit(extract_pdf_job)

print("Adobe extraction job submitted successfully.")
print("Job location:", location)

Adobe extraction job submitted successfully.
Job location: https://pdf-services-ue1.adobe.io/operation/extractpdf/d4844abd-88b9-4eb4-af29-90125a2304dc/status


In [12]:
# ============================================
# MIRACL - Get Adobe Extraction Result
# ============================================

extract_pdf_result = pdf_services.get_job_result(
    location,
    ExtractPDFResult,
)

print("Adobe extraction completed successfully.")
print("Extraction result received.")

Adobe extraction completed successfully.
Extraction result received.


In [13]:
# ============================================
# MIRACL - Download Adobe Structured Data
# ============================================

result_asset = extract_pdf_result.get_result().get_resource()

zip_bytes = requests.get(
    result_asset.get_download_uri(),
).content

output_zip = Path("adobe_extraction_result.zip")

with open(output_zip, "wb") as file:
    file.write(zip_bytes)

print("Adobe extraction ZIP downloaded successfully.")
print("Saved as:", output_zip)
print("File size:", round(output_zip.stat().st_size / 1024, 2), "KB")

Adobe extraction ZIP downloaded successfully.
Saved as: adobe_extraction_result.zip
File size: 5.6 KB


In [14]:
# ============================================
# MIRACL - Extract Adobe Output
# ============================================

extraction_dir = Path("adobe_extraction")

with zipfile.ZipFile(output_zip, "r") as zip_ref:
    zip_ref.extractall(extraction_dir)

print("Adobe extraction output unpacked successfully.")
print("Files extracted:")

for file in extraction_dir.rglob("*"):
    if file.is_file():
        print(" -", file)

Adobe extraction output unpacked successfully.
Files extracted:
 - adobe_extraction/structuredData.json


In [15]:
# ============================================
# MIRACL - Inspect Adobe Structured Data
# ============================================

structured_data_path = extraction_dir / "structuredData.json"

with open(structured_data_path, "r", encoding="utf-8") as file:
    structured_data = json.load(file)

print("structuredData.json loaded successfully.")
print("Top-level keys:")
print(list(structured_data.keys()))

structuredData.json loaded successfully.
Top-level keys:
['version', 'extended_metadata', 'elements', 'pages']


In [16]:
# ============================================
# MIRACL - Inspect First Adobe Element
# ============================================

print(json.dumps(structured_data["elements"][0], indent=2))

{
  "Bounds": [
    131.98199462890625,
    789.6554412841797,
    469.5079345703125,
    818.1358184814453
  ],
  "Font": {
    "alt_family_name": "CMB Extra",
    "embedded": true,
    "encoding": "Custom",
    "family_name": "CMB Extra 12",
    "font_type": "Type1",
    "italic": false,
    "monospaced": false,
    "name": "CSKFON+CMBX12",
    "subset": true,
    "weight": 400
  },
  "HasClip": false,
  "ObjectID": 61,
  "Page": 0,
  "Path": "//Document/Sect/Title",
  "Text": "Muhammad Thameem S A ",
  "TextSize": 24.787094116210938,
  "attributes": {
    "LineHeight": 29.75,
    "TextAlign": "Center"
  }
}


In [17]:
# ============================================
# MIRACL - Inspect First 10 Adobe Elements
# ============================================

for i, element in enumerate(structured_data["elements"][:10]):
    print(f"\n--- Element {i} ---")
    print("Text:", element.get("Text"))
    print("Page:", element.get("Page"))
    print("Path:", element.get("Path"))


--- Element 0 ---
Text: Muhammad Thameem S A 
Page: 0
Path: //Document/Sect/Title

--- Element 1 ---
Text: +91 9037113737 
Page: 0
Path: //Document/Sect/P

--- Element 2 ---
Text: | Bangalore, India 
Page: 0
Path: //Document/Sect/L/LI/LBody

--- Element 3 ---
Text: | (<mailto:muhammadthameem2003@gmail.com>)muhammadthameem2003@gmail.com | 
Page: 0
Path: //Document/Sect/L/LI[2]/LBody/ParagraphSpan

--- Element 4 ---
Text: | (<https://github.com/muhammadthameem>)github.com/muhammadthameem 
Page: 0
Path: //Document/Sect/L/LI[2]/LBody/L/LI/LBody

--- Element 5 ---
Text: (<https://linkedin.com/in/muhammadthameemsa>)linkedin.com/in/muhammadthameemsa 
Page: 0
Path: //Document/Sect/L/LI[3]/LBody/Reference

--- Element 6 ---
Text: Profile 
Page: 0
Path: //Document/Sect[2]/H1

--- Element 7 ---
Text: Recent B.Tech graduate in Artificial Intelligence & Data Science with hands-on experience in data analysis, SQL, Excel, Python, and Power BI. Skilled in extracting, cleaning, and transforming raw da

In [18]:
# ============================================
# MIRACL - Extract Basic Metadata
# ============================================

metadata_records = []

for element in structured_data["elements"]:
    text = element.get("Text", "").strip()
    path = element.get("Path", "")
    page = element.get("Page", None)

    if not text:
        continue

    # Determine element type from Adobe's structural path
    if "/Title" in path:
        element_type = "title"
    elif "/H1" in path:
        element_type = "heading"
    elif "/H2" in path:
        element_type = "subheading"
    elif "/P" in path:
        element_type = "paragraph"
    elif "/L/" in path or "/LBody" in path:
        element_type = "list"
    else:
        element_type = "other"

    metadata_records.append({
        "text": text,
        "page": page,
        "element_type": element_type,
        "path": path,
    })

print("Metadata extraction completed.")
print("Total metadata records:", len(metadata_records))

print("\nFirst 5 metadata records:")
for record in metadata_records[:5]:
    print(record)

Metadata extraction completed.
Total metadata records: 62

First 5 metadata records:
{'text': 'Muhammad Thameem S A', 'page': 0, 'element_type': 'title', 'path': '//Document/Sect/Title'}
{'text': '+91 9037113737', 'page': 0, 'element_type': 'paragraph', 'path': '//Document/Sect/P'}
{'text': '| Bangalore, India', 'page': 0, 'element_type': 'list', 'path': '//Document/Sect/L/LI/LBody'}
{'text': '| (<mailto:muhammadthameem2003@gmail.com>)muhammadthameem2003@gmail.com |', 'page': 0, 'element_type': 'paragraph', 'path': '//Document/Sect/L/LI[2]/LBody/ParagraphSpan'}
{'text': '| (<https://github.com/muhammadthameem>)github.com/muhammadthameem', 'page': 0, 'element_type': 'list', 'path': '//Document/Sect/L/LI[2]/LBody/L/LI/LBody'}


In [19]:
# ============================================
# MIRACL - Metadata Element Distribution
# ============================================

from collections import Counter

element_type_counts = Counter(
    record["element_type"]
    for record in metadata_records
)

print("Metadata element distribution:\n")

for element_type, count in element_type_counts.items():
    print(f"{element_type}: {count}")

Metadata element distribution:

title: 1
paragraph: 7
list: 47
heading: 5
subheading: 2


In [20]:
# ============================================
# MIRACL - Inspect Document Headings
# ============================================

print("Document headings:\n")

for record in metadata_records:
    if record["element_type"] in ["heading", "subheading"]:
        print(
            f"{record['element_type'].upper():12} | "
            f"{record['text']} | "
            f"{record['path']}"
        )

Document headings:

HEADING      | Profile | //Document/Sect[2]/H1
HEADING      | Technical Skills | //Document/Sect[3]/H1
HEADING      | Education | //Document/Sect[4]/H1
HEADING      | Projects | //Document/Sect[5]/H1
SUBHEADING   | Nike Sales Analysis | //Document/Sect[5]/Sect/H2
SUBHEADING   | Frozen Food Wholesale & Retail ERP System | //Document/Sect[5]/Sect[2]/H2
HEADING      | Certifications & Hackathons | //Document/Sect[6]/H1


In [21]:
# ============================================
# MIRACL - Add Section & Subsection Context
# ============================================

contextual_records = []

current_section = None
current_subsection = None

for record in metadata_records:
    element_type = record["element_type"]
    text = record["text"]

    if element_type == "heading":
        current_section = text
        current_subsection = None

    elif element_type == "subheading":
        current_subsection = text

    contextual_records.append({
        "text": text,
        "page": record["page"],
        "element_type": element_type,
        "section": current_section,
        "subsection": current_subsection,
        "path": record["path"],
    })

print("Contextual metadata created.")
print("Total records:", len(contextual_records))

print("\nFirst 10 contextual records:\n")

for record in contextual_records[:10]:
    print(record)

Contextual metadata created.
Total records: 62

First 10 contextual records:

{'text': 'Muhammad Thameem S A', 'page': 0, 'element_type': 'title', 'section': None, 'subsection': None, 'path': '//Document/Sect/Title'}
{'text': '+91 9037113737', 'page': 0, 'element_type': 'paragraph', 'section': None, 'subsection': None, 'path': '//Document/Sect/P'}
{'text': '| Bangalore, India', 'page': 0, 'element_type': 'list', 'section': None, 'subsection': None, 'path': '//Document/Sect/L/LI/LBody'}
{'text': '| (<mailto:muhammadthameem2003@gmail.com>)muhammadthameem2003@gmail.com |', 'page': 0, 'element_type': 'paragraph', 'section': None, 'subsection': None, 'path': '//Document/Sect/L/LI[2]/LBody/ParagraphSpan'}
{'text': '| (<https://github.com/muhammadthameem>)github.com/muhammadthameem', 'page': 0, 'element_type': 'list', 'section': None, 'subsection': None, 'path': '//Document/Sect/L/LI[2]/LBody/L/LI/LBody'}
{'text': '(<https://linkedin.com/in/muhammadthameemsa>)linkedin.com/in/muhammadthameemsa

In [22]:
# ============================================
# MIRACL - Verify Section/Subsection Context
# ============================================

for record in contextual_records:
    if record["section"] == "Projects":
        print(
            f"{record['element_type'].upper():12} | "
            f"Section: {record['section']} | "
            f"Subsection: {record['subsection']} | "
            f"Text: {record['text'][:100]}"
        )

HEADING      | Section: Projects | Subsection: None | Text: Projects
PARAGRAPH    | Section: Projects | Subsection: None | Text: MIRACL – End-to-End RAG Pipeline Python, Transformers, OpenAI API, LangChain, LlamaIndex, FAISS | (<
LIST         | Section: Projects | Subsection: None | Text: –
LIST         | Section: Projects | Subsection: None | Text: Built an end-to-end Retrieval-Augmented Generation (RAG) system, from raw document ingestion to a wo
LIST         | Section: Projects | Subsection: None | Text: –
LIST         | Section: Projects | Subsection: None | Text: Used LangChain and LlamaIndex to develop the core retrieval and orchestration functionality, with FA
LIST         | Section: Projects | Subsection: None | Text: –
LIST         | Section: Projects | Subsection: None | Text: Evaluated generated responses for accuracy and groundedness against source documents, iterating on r
SUBHEADING   | Section: Projects | Subsection: Nike Sales Analysis | Text: Nike Sales Analysis
PARAGR

In [23]:
# ============================================
# MIRACL - Create Contextual Chunks
# ============================================

chunks = []

current_chunk = []
current_section = None
current_subsection = None


def save_chunk():
    if not current_chunk:
        return

    chunk_text = "\n".join(current_chunk).strip()

    if not chunk_text:
        return

    chunks.append({
        "text": chunk_text,
        "section": current_section,
        "subsection": current_subsection,
    })


for record in contextual_records:

    element_type = record["element_type"]
    text = record["text"]

    # New main section
    if element_type == "heading":

        save_chunk()

        current_chunk = []
        current_section = text
        current_subsection = None

        current_chunk.append(text)

    # New subsection
    elif element_type == "subheading":

        save_chunk()

        current_chunk = []
        current_subsection = text

        current_chunk.append(text)

    # Normal content
    else:
        current_chunk.append(text)


# Save final chunk
save_chunk()


print("Contextual chunking completed.")
print("Total chunks:", len(chunks))

print("\nFirst 5 chunks:\n")

for i, chunk in enumerate(chunks[:5]):
    print(f"--- Chunk {i} ---")
    print("Section:", chunk["section"])
    print("Subsection:", chunk["subsection"])
    print("Text:")
    print(chunk["text"][:500])
    print()

Contextual chunking completed.
Total chunks: 8

First 5 chunks:

--- Chunk 0 ---
Section: None
Subsection: None
Text:
Muhammad Thameem S A
+91 9037113737
| Bangalore, India
| (<mailto:muhammadthameem2003@gmail.com>)muhammadthameem2003@gmail.com |
| (<https://github.com/muhammadthameem>)github.com/muhammadthameem
(<https://linkedin.com/in/muhammadthameemsa>)linkedin.com/in/muhammadthameemsa

--- Chunk 1 ---
Section: Profile
Subsection: None
Text:
Profile
Recent B.Tech graduate in Artificial Intelligence & Data Science with hands-on experience in data analysis, SQL, Excel, Python, and Power BI. Skilled in extracting, cleaning, and transforming raw data into clear, actionable business insights, building interactive dashboards, and applying statistical thinking to identify trends and root causes. Comfortable working with large, complex datasets end-to-end and communicating findings clearly to non-technical stakeholders. Detail-oriented, ana

--- Chunk 2 ---
Section: Technical Skills
Subsec

In [24]:
# ============================================
# MIRACL - Inspect All Contextual Chunks
# ============================================

for i, chunk in enumerate(chunks):

    print(f"\n{'=' * 70}")
    print(f"CHUNK {i}")
    print(f"{'=' * 70}")

    print("Section:", chunk["section"])
    print("Subsection:", chunk["subsection"])
    print("\nText:")
    print(chunk["text"])


CHUNK 0
Section: None
Subsection: None

Text:
Muhammad Thameem S A
+91 9037113737
| Bangalore, India
| (<mailto:muhammadthameem2003@gmail.com>)muhammadthameem2003@gmail.com |
| (<https://github.com/muhammadthameem>)github.com/muhammadthameem
(<https://linkedin.com/in/muhammadthameemsa>)linkedin.com/in/muhammadthameemsa

CHUNK 1
Section: Profile
Subsection: None

Text:
Profile
Recent B.Tech graduate in Artificial Intelligence & Data Science with hands-on experience in data analysis, SQL, Excel, Python, and Power BI. Skilled in extracting, cleaning, and transforming raw data into clear, actionable business insights, building interactive dashboards, and applying statistical thinking to identify trends and root causes. Comfortable working with large, complex datasets end-to-end and communicating findings clearly to non-technical stakeholders. Detail-oriented, analytically strong, and looking to bring a data-driven, problem-solving mindset to a Data Analyst role.

CHUNK 2
Section: Technica

In [25]:
# ============================================
# MIRACL - Load BGE Embedding Model
# ============================================

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("BGE embedding model loaded successfully.")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE embedding model loaded successfully.
Embedding dimension: 384


/var/folders/t7/cy18_mh15nq45ylrlc29bybr0000gn/T/ipykernel_22672/4016067974.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [26]:
# ============================================
# MIRACL - Generate Chunk Embeddings
# ============================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print("Chunk embeddings generated successfully.")
print("Number of embeddings:", len(chunk_embeddings))
print("Embedding dimension:", chunk_embeddings.shape[1])

Chunk embeddings generated successfully.
Number of embeddings: 8
Embedding dimension: 384


In [27]:
# ============================================
# MIRACL - Initialize ChromaDB
# ============================================

chroma_path = Path("chroma_db")
chroma_path.mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=str(chroma_path)
)

collection = chroma_client.get_or_create_collection(
    name="miracl_documents_contextual"
)

print("ChromaDB initialized successfully.")
print("Database path:", chroma_path)
print("Collection:", collection.name)
print("Existing documents:", collection.count())

ChromaDB initialized successfully.
Database path: chroma_db
Collection: miracl_documents_contextual
Existing documents: 0


In [28]:
# ============================================
# MIRACL - Store Chunks in ChromaDB
# ============================================

documents = []
metadatas = []
ids = []

for i, chunk in enumerate(chunks):

    documents.append(chunk["text"])

    metadatas.append({
        "section": chunk["section"] or "",
        "subsection": chunk["subsection"] or "",
        "chunk_id": i,
    })

    ids.append(f"chunk_{i}")

collection.add(
    ids=ids,
    documents=documents,
    embeddings=chunk_embeddings.tolist(),
    metadatas=metadatas,
)

print("Chunks stored in ChromaDB successfully.")
print("Documents stored:", collection.count())

Chunks stored in ChromaDB successfully.
Documents stored: 8


In [34]:
# ============================================
# MIRACL - Semantic Retrieval Test
# ============================================

query = "What projects did I work on?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
)

print("Query:", query)
print("\nRetrieved chunks:\n")

for i, document in enumerate(results["documents"][0]):
    print(f"--- Result {i + 1} ---")
    print("Distance:", results["distances"][0][i])
    print("Section:", results["metadatas"][0][i]["section"])
    print("Subsection:", results["metadatas"][0][i]["subsection"])
    print("Text:", document[:500])
    print()

Query: What projects did I work on?

Retrieved chunks:

--- Result 1 ---
Distance: 0.8036662340164185
Section: Projects
Subsection: 
Text: Projects
MIRACL – End-to-End RAG Pipeline Python, Transformers, OpenAI API, LangChain, LlamaIndex, FAISS | (<https://github.com/Muhammadthameem/MIRACL>)GitHub
–
Built an end-to-end Retrieval-Augmented Generation (RAG) system, from raw document ingestion to a working question-answering pipeline powered by an LLM.
–
Used LangChain and LlamaIndex to develop the core retrieval and orchestration functionality, with FAISS for scalable vector embedding search.
–
Evaluated generated responses for accura

--- Result 2 ---
Distance: 0.8757785558700562
Section: Technical Skills
Subsection: 
Text: Technical Skills
–
Programming: Python (Pandas, NumPy, Scikit-learn)
–
SQL: Query writing & optimization, joins, aggregations, data extraction, filtering, reconciliation
–
Data Visualization & BI: Power BI (interactive dashboards, KPI reporting, report design), Matplo

In [35]:
# ============================================
# MIRACL - Semantic Retrieval Test
# ============================================

query = "What did I do in the Nike Sales Analysis project?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
)

print("Query:", query)
print("\nRetrieved chunks:\n")

for i, document in enumerate(results["documents"][0]):
    print(f"--- Result {i + 1} ---")
    print("Distance:", results["distances"][0][i])
    print("Section:", results["metadatas"][0][i]["section"])
    print("Subsection:", results["metadatas"][0][i]["subsection"])
    print("Text:", document)
    print()

Query: What did I do in the Nike Sales Analysis project?

Retrieved chunks:

--- Result 1 ---
Distance: 0.47335323691368103
Section: Projects
Subsection: Nike Sales Analysis
Text: Nike Sales Analysis
Power BI, Python (Pandas), DAX | (<https://github.com/Muhammadthameem/nike-sales-analysis-powerbi>)GitHub
–
Analyzed a 3,000-record Nike retail sales dataset using Python and Power BI to identify revenue trends, regional and category performance, customer age-group behavior, and discount impact.
–
Cleaned and validated the underlying data in Pandas, correcting pricing inconsistencies and structuring it into a fact and dimension model before analysis and reporting.
–
Designed and built an interactive Power BI dashboard with DAX-driven KPIs, year-over-year comparisons, and an image-based product slicer to support business decision-making.

--- Result 2 ---
Distance: 0.9132272005081177
Section: Projects
Subsection: 
Text: Projects
MIRACL – End-to-End RAG Pipeline Python, Transformers, OpenAI 

In [37]:
# ============================================
# MIRACL - Configure LlamaIndex Embeddings
# ============================================

llama_embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("LlamaIndex BGE embedding model configured successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LlamaIndex BGE embedding model configured successfully.


In [39]:
# ============================================
# MIRACL - Connect ChromaDB with LlamaIndex
# ============================================

chroma_vector_store = ChromaVectorStore(
    chroma_collection=collection
)

llama_index = VectorStoreIndex.from_vector_store(
    vector_store=chroma_vector_store,
    embed_model=llama_embed_model,
)

print("ChromaDB connected to LlamaIndex successfully.")

ChromaDB connected to LlamaIndex successfully.


In [41]:
# ============================================
# MIRACL - LlamaIndex Retriever Test
# ============================================

retriever = llama_index.as_retriever(
    similarity_top_k=3
)

query = "What did I do in the Nike Sales Analysis project?"

retrieved_nodes = retriever.retrieve(query)

print("Query:", query)
print("\nRetrieved nodes:\n")

for i, node in enumerate(retrieved_nodes):
    print(f"--- Result {i + 1} ---")
    print("Score:", node.score)
    print("Text:", node.text)
    print("Metadata:", node.metadata)
    print()

Query: What did I do in the Nike Sales Analysis project?

Retrieved nodes:

--- Result 1 ---
Score: 0.6105498535787577
Text: Nike Sales Analysis
Power BI, Python (Pandas), DAX | (<https://github.com/Muhammadthameem/nike-sales-analysis-powerbi>)GitHub
–
Analyzed a 3,000-record Nike retail sales dataset using Python and Power BI to identify revenue trends, regional and category performance, customer age-group behavior, and discount impact.
–
Cleaned and validated the underlying data in Pandas, correcting pricing inconsistencies and structuring it into a fact and dimension model before analysis and reporting.
–
Designed and built an interactive Power BI dashboard with DAX-driven KPIs, year-over-year comparisons, and an image-based product slicer to support business decision-making.
Metadata: {'subsection': 'Nike Sales Analysis', 'chunk_id': 5, 'section': 'Projects'}

--- Result 2 ---
Score: 0.4331915501508789
Text: Projects
MIRACL – End-to-End RAG Pipeline Python, Transformers, OpenAI API

In [42]:
# ============================================
# MIRACL - Check Ollama
# ============================================

import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

NAME           ID              SIZE      MODIFIED     
llama3.2:3b    a80c4f17acd5    2.0 GB    24 hours ago    



In [43]:
# ============================================
# MIRACL - Check Ollama
# ============================================

import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

NAME           ID              SIZE      MODIFIED     
llama3.2:3b    a80c4f17acd5    2.0 GB    24 hours ago    



In [44]:
# ============================================
# MIRACL - Test Ollama LLM
# ============================================

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0,
)

response = llm.invoke(
    "Explain in one sentence what semantic search means."
)

print(response.content)

Semantic search refers to the use of natural language processing (NLP) and machine learning algorithms to understand the context, intent, and meaning behind a user's search query, providing more accurate and relevant results.


In [45]:
# ============================================
# MIRACL - Build RAG Context
# ============================================

query = "What did I do in the Nike Sales Analysis project?"

retrieved_nodes = retriever.retrieve(query)

context_parts = []

for i, node in enumerate(retrieved_nodes):
    context_parts.append(
        f"[Context {i + 1}]\n"
        f"Section: {node.metadata.get('section', '')}\n"
        f"Subsection: {node.metadata.get('subsection', '')}\n"
        f"Content:\n{node.text}"
    )

context = "\n\n".join(context_parts)

print("QUERY:")
print(query)

print("\n" + "=" * 70)
print("RETRIEVED CONTEXT")
print("=" * 70)

print(context)

QUERY:
What did I do in the Nike Sales Analysis project?

RETRIEVED CONTEXT
[Context 1]
Section: Projects
Subsection: Nike Sales Analysis
Content:
Nike Sales Analysis
Power BI, Python (Pandas), DAX | (<https://github.com/Muhammadthameem/nike-sales-analysis-powerbi>)GitHub
–
Analyzed a 3,000-record Nike retail sales dataset using Python and Power BI to identify revenue trends, regional and category performance, customer age-group behavior, and discount impact.
–
Cleaned and validated the underlying data in Pandas, correcting pricing inconsistencies and structuring it into a fact and dimension model before analysis and reporting.
–
Designed and built an interactive Power BI dashboard with DAX-driven KPIs, year-over-year comparisons, and an image-based product slicer to support business decision-making.

[Context 2]
Section: Projects
Subsection: 
Content:
Projects
MIRACL – End-to-End RAG Pipeline Python, Transformers, OpenAI API, LangChain, LlamaIndex, FAISS | (<https://github.com/Muhamma

In [47]:
# ============================================
# MIRACL - Create RAG Prompt
# ============================================

prompt_template = ChatPromptTemplate.from_template("""
You are a document question-answering assistant.

Answer the user's question using ONLY the retrieved context below.

Rules:
- Use only information explicitly supported by the retrieved context.
- Do not add facts from your own knowledge.
- Do not assume that related phrases mean the same thing.
- If the exact answer is not explicitly supported by the context, say:
  "The information is not explicitly stated in the retrieved context."

Retrieved context:
{context}

User question:
{question}

Answer:
""")

prompt = prompt_template.invoke({
    "context": context,
    "question": query,
})

print(prompt.to_string())

Human: 
You are a document question-answering assistant.

Answer the user's question using ONLY the retrieved context below.

Rules:
- Use only information explicitly supported by the retrieved context.
- Do not add facts from your own knowledge.
- Do not assume that related phrases mean the same thing.
- If the exact answer is not explicitly supported by the context, say:
  "The information is not explicitly stated in the retrieved context."

Retrieved context:
[Context 1]
Section: Projects
Subsection: Nike Sales Analysis
Content:
Nike Sales Analysis
Power BI, Python (Pandas), DAX | (<https://github.com/Muhammadthameem/nike-sales-analysis-powerbi>)GitHub
–
Analyzed a 3,000-record Nike retail sales dataset using Python and Power BI to identify revenue trends, regional and category performance, customer age-group behavior, and discount impact.
–
Cleaned and validated the underlying data in Pandas, correcting pricing inconsistencies and structuring it into a fact and dimension model befo

In [48]:
# ============================================
# MIRACL - Generate Grounded RAG Answer
# ============================================

response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\n" + "=" * 70)
print("MIRACL ANSWER")
print("=" * 70)

print(response.content)

QUESTION:
What did I do in the Nike Sales Analysis project?

MIRACL ANSWER
The information is not explicitly stated in the retrieved context.


In [49]:
# ============================================
# MIRACL - Refined RAG Prompt
# ============================================

prompt_template = ChatPromptTemplate.from_template("""
You are answering questions about a document.

Use the retrieved context to answer the user's question.
Only include information that is explicitly present in the context.
Do not use outside knowledge or invent information.

Retrieved context:
{context}

Question:
{question}

Provide a concise answer based directly on the retrieved context.
""")

prompt = prompt_template.invoke({
    "context": context,
    "question": query,
})

response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\n" + "=" * 70)
print("MIRACL ANSWER")
print("=" * 70)

print(response.content)

QUESTION:
What did I do in the Nike Sales Analysis project?

MIRACL ANSWER
You analyzed a 3,000-record Nike retail sales dataset using Python and Power BI to identify revenue trends, regional and category performance, customer age-group behavior, and discount impact.


In [50]:
# ============================================
# MIRACL - Test Grounding / Unknown Question
# ============================================

query = "What is my favorite programming language?"

retrieved_nodes = retriever.retrieve(query)

context_parts = []

for i, node in enumerate(retrieved_nodes):
    context_parts.append(
        f"[Context {i + 1}]\n"
        f"Section: {node.metadata.get('section', '')}\n"
        f"Subsection: {node.metadata.get('subsection', '')}\n"
        f"Content:\n{node.text}"
    )

context = "\n\n".join(context_parts)

prompt = prompt_template.invoke({
    "context": context,
    "question": query,
})

response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\n" + "=" * 70)
print("MIRACL ANSWER")
print("=" * 70)

print(response.content)

QUESTION:
What is my favorite programming language?

MIRACL ANSWER
There is no information about your favorite programming language in the retrieved context.


In [51]:
# ============================================
# MIRACL - Second RAG Test
# ============================================

query = "What technologies did I use for the Frozen Food ERP project?"

retrieved_nodes = retriever.retrieve(query)

context_parts = []

for i, node in enumerate(retrieved_nodes):
    context_parts.append(
        f"[Context {i + 1}]\n"
        f"Section: {node.metadata.get('section', '')}\n"
        f"Subsection: {node.metadata.get('subsection', '')}\n"
        f"Content:\n{node.text}"
    )

context = "\n\n".join(context_parts)

prompt = prompt_template.invoke({
    "context": context,
    "question": query,
})

response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\n" + "=" * 70)
print("MIRACL ANSWER")
print("=" * 70)

print(response.content)

QUESTION:
What technologies did I use for the Frozen Food ERP project?

MIRACL ANSWER
According to the retrieved context, the technologies used for the Frozen Food ERP project are:

- Next.js
- React
- TypeScript
- Prisma
- SQLite
- Electron
